# PDF Parsing with olmOCR-2-7B-1025-FP8

Parses every PDF under `data/textbook/` with [allenai/olmOCR-2-7B-1025-FP8](https://huggingface.co/allenai/olmOCR-2-7B-1025-FP8) — an FP8 (F8_E4M3) quantized build of the 7B Qwen2.5-VL-based VLM fine-tuned on the olmOCR-mix-1025 dataset (+ GRPO RL), specialized for OCR of dense text, math (LaTeX), and tables (HTML). Benchmarked at 82.4 ± 1.1 overall vs 82.3 ± 1.1 for the unquantized BF16 model — essentially no accuracy loss for roughly half the weight size.

Much lighter than Infinity-Parser2-Pro (35B): **7B in FP8 is ~8GB of weights**, fits on a single consumer/prosumer GPU (8GB+ VRAM recommended, 12GB+ comfortable with KV cache/activations headroom).
- **CUDA only.** FP8 (F8_E4M3) tensors require CUDA compute capability 8.9+ (Ada/Hopper, e.g. RTX 4000-series, L4, H100) for native FP8 kernels; on older CUDA GPUs `transformers` will upcast and you lose the memory savings. There is no MPS/Apple Silicon path for this checkpoint — use the BF16 [allenai/olmOCR-2-7B-1025](https://huggingface.co/allenai/olmOCR-2-7B-1025) instead on Mac.
- No `flash_attention_2` unless installed separately (CUDA-only kernel).

Recommended deps (not in this project's `pyproject.toml` — install separately, e.g. into a `.venv-olmocr` venv):
```
uv venv .venv-olmocr --python 3.13
uv pip install --python .venv-olmocr \
    torch --index-url https://download.pytorch.org/whl/cu124 \
    "olmocr>=0.4.0" transformers accelerate qwen-vl-utils pillow pymupdf tqdm pyyaml
# optional, for max throughput: flash-attn, vllm (see allenai/olmocr toolkit)
```

Unlike Infinity-Parser2-Pro's JSON-layout mode, olmOCR outputs **Markdown directly** (equations as LaTeX, tables as HTML) with a **YAML front-matter block** on top (`primary_language`, `is_rotation_valid`, `rotation_correction`, `is_table`, `is_diagram`). It does not emit bboxes, so there's no crop-visuals step here — figures are referenced inline via markdown image syntax by the model itself (`![alt](page_startx_starty_width_height.png)`) but not materialized as real crops. Output is saved standalone under `output/<pdf_stem>/olmocr/`, following the same per-PDF layout convention as the other parser notebooks.

This follows the official `allenai/olmocr` toolkit's prompt (`build_no_anchoring_v4_yaml_prompt`) and inference conventions (longest-side 1288px render, `max_tokens=8000`, `temperature=0.0`) — see [github.com/allenai/olmocr](https://github.com/allenai/olmocr) and the [model card](https://huggingface.co/allenai/olmOCR-2-7B-1025-FP8). For production-scale throughput, prefer the official toolkit's vLLM-based pipeline over this notebook's per-page `transformers` loop.

## Setup

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent  # notebook lives in notebooks/

TEXTBOOK_DIR = PROJECT_ROOT / "data" / "textbook"
OUTPUT_ROOT = PROJECT_ROOT / "output"

pdf_paths = sorted(TEXTBOOK_DIR.rglob("*.pdf"))
assert pdf_paths, f"No PDFs found under {TEXTBOOK_DIR}"

len(pdf_paths), pdf_paths[0]

## Render PDF pages to images

Rasterizes each PDF page to PNG via PyMuPDF, scaled so the **longest side is `TARGET_LONGEST_DIM` pixels (1288, olmOCR's convention)** rather than a fixed DPI — matches how the model was trained/evaluated.

In [ ]:
import fitz  # PyMuPDF

TARGET_LONGEST_DIM = 1288  # olmOCR convention: render longest page side to this many px


def render_pages(pdf_path: Path, output_dir: Path, target_longest_dim: int = TARGET_LONGEST_DIM) -> list[Path]:
    """Renders every page of `pdf_path` to a PNG in `output_dir`, named
    `page_{page_idx:04d}.png` (0-indexed), scaled so the longest side is
    `target_longest_dim` pixels. Returns image paths in page order."""
    output_dir.mkdir(parents=True, exist_ok=True)

    paths = []
    with fitz.open(pdf_path) as doc:
        for page_idx, page in enumerate(doc):
            longest_side = max(page.rect.width, page.rect.height)
            zoom = target_longest_dim / longest_side
            pix = page.get_pixmap(matrix=fitz.Matrix(zoom, zoom))
            image_path = output_dir / f"page_{page_idx:04d}.png"
            pix.save(image_path)
            paths.append(image_path)

    return paths

## Load olmOCR-2-7B-1025-FP8

Base architecture is Qwen2.5-VL-7B, loaded via `Qwen2_5_VLForConditionalGeneration` (per the model card) rather than a generic `AutoModelForImageTextToText` — the model repo doesn't set `trust_remote_code` custom classes. Weights are pre-quantized to FP8 (F8_E4M3), so **CUDA with compute capability 8.9+ (Ada/Hopper) is required** — there is no MPS fallback for this checkpoint.

In [ ]:
import torch
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA device found. olmOCR-2-7B-1025-FP8 requires a CUDA GPU with "
        "compute capability 8.9+ (Ada/Hopper, e.g. RTX 4000-series, L4, H100) "
        "for native FP8 (F8_E4M3) kernels. Use allenai/olmOCR-2-7B-1025 (BF16) "
        "instead on Apple Silicon / MPS."
    )

DEVICE = "cuda"
MODEL_PATH = "allenai/olmOCR-2-7B-1025-FP8"

processor = AutoProcessor.from_pretrained(MODEL_PATH)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_PATH,
    device_map="auto",
    # attn_implementation="flash_attention_2",  # optional, requires flash-attn install
)
model.eval()

## Parse function

Uses a variant of olmOCR's official "no-anchoring v4 YAML" prompt (`build_no_anchoring_v4_yaml_prompt` from the [allenai/olmocr toolkit](https://github.com/allenai/olmocr)), with an added instruction to preserve original text structure (headings, paragraph breaks, list items, multi-column reading order) instead of the default's natural-reading reflow. Output is YAML front matter (`---` delimited) followed by Markdown body (equations as LaTeX, tables as HTML). We parse the front matter to get `natural_text` (the actual page content) plus metadata (language, rotation, table/diagram flags), matching `PageResponse` in the toolkit.

Inference params (`max_new_tokens=8000`, `temperature=0.0`, greedy) match the toolkit's default `build_page_query`.

In [ ]:
import yaml
from PIL import Image
from qwen_vl_utils import process_vision_info

NO_ANCHORING_V4_YAML_PROMPT = (
    "Attached is one page of a document that you must process. "
    "Just return the plain text representation of this document as if you were reading it naturally. "
    "Convert equations to LateX and tables to HTML.\n"
    "Preserve the original text structure: keep headings, paragraph breaks, list items, and "
    "reading order (including multi-column layouts, read left-to-right column by column) as they "
    "appear on the page. Do not merge separate paragraphs or reflow line/paragraph breaks.\n"
    "Remove page headers and footers (e.g. running titles, page numbers), but keep references "
    "and footnotes.\n"
    "If there are any figures or charts, label them with the following markdown syntax "
    "![Alt text describing the contents of the figure](page_startx_starty_width_height.png)\n"
    "Return your output as markdown, with a front matter section on top specifying values for the "
    "primary_language, is_rotation_valid, rotation_correction, is_table, and is_diagram parameters."
)


def extract_front_matter_and_text(markdown_content: str) -> tuple[dict, str]:
    """Mirrors olmocr.train.front_matter.FrontMatterParser._extract_front_matter_and_text:
    splits a `---`-delimited YAML front matter block from the markdown body."""
    if markdown_content.startswith("---\n"):
        end_index = markdown_content.find("\n---", 4)
        if end_index != -1:
            front_matter_str = markdown_content[4:end_index]
            text = markdown_content[end_index + 4:].strip()
            return yaml.safe_load(front_matter_str) or {}, text
    return {}, markdown_content.strip()


def parse_page(image_path: Path, max_new_tokens: int = 8000) -> tuple[dict, str]:
    """Returns (front_matter, natural_text) for one page image, following the
    official olmOCR toolkit's prompt and greedy-decoding inference params."""
    page_image = Image.open(image_path).convert("RGB")
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": page_image},
                {"type": "text", "text": NO_ANCHORING_V4_YAML_PROMPT},
            ],
        }
    ]

    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False, temperature=None
        )

    generated_trimmed = [
        out_ids[len(in_ids):]
        for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    raw = processor.batch_decode(
        generated_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0].strip()

    front_matter, natural_text = extract_front_matter_and_text(raw)
    return front_matter, natural_text

## Smoke test on one PDF, one page

In [ ]:
sample_pdf = pdf_paths[0]
sample_out_dir = OUTPUT_ROOT / sample_pdf.stem / "olmocr"
sample_pages = render_pages(sample_pdf, sample_out_dir / "rendered_pages")

sample_front_matter, sample_text = parse_page(sample_pages[0])
print(sample_front_matter)
print()
print(sample_text)

## Batch: parse every PDF under `data/textbook/`

For each PDF: render pages, parse each page with olmOCR, concatenate per-page `natural_text` into one `.md` file, and dump per-page front matter (metadata) to a sidecar `.jsonl`. Skips PDFs whose output already exists, so this cell is safe to re-run after an interruption.

In [ ]:
import json

from tqdm.auto import tqdm

PAGE_BREAK = "\n\n---\n\n"

for pdf_path in tqdm(pdf_paths, desc="PDFs"):
    out_dir = OUTPUT_ROOT / pdf_path.stem / "olmocr"
    out_md_path = out_dir / f"{pdf_path.stem}.md"
    if out_md_path.exists():
        continue

    page_image_paths = render_pages(pdf_path, out_dir / "rendered_pages")

    page_texts = []
    page_front_matters = []
    for page_idx, image_path in enumerate(tqdm(page_image_paths, desc=pdf_path.stem, leave=False)):
        front_matter, text = parse_page(image_path)
        page_texts.append(text or "")
        page_front_matters.append({"page": page_idx, **front_matter})

    out_md_path.write_text(PAGE_BREAK.join(page_texts), encoding="utf-8")

    out_meta_path = out_dir / f"{pdf_path.stem}.jsonl"
    with out_meta_path.open("w", encoding="utf-8") as f:
        for meta in page_front_matters:
            f.write(json.dumps(meta) + "\n")